# Phase 5: Evaluation

**CRISP-DM Phase Description:**  
At this stage, the model(s) appear to have high quality from a technical standpoint. Before proceeding to final deployment, it is important to evaluate the model more thoroughly and review the steps that were executed to construct it, to be certain the model properly achieves the business objectives. A key objective is to determine if there is some important business issue that has not been sufficiently considered.

---

In [8]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

---
### Task 1: Evaluate Results

Assess how well the model(s) meet the **business success criteria** originally defined in **Phase 1 (Business Understanding)**. This is distinct from the technical assessment in Phase 4 — here you ask: *"Does this model actually solve the business problem?"*

Key activities:
- **Map Technical Metrics to Business Criteria:** Compare the model's performance metrics (from Phase 4) against the business success criteria and data mining success criteria defined in Phase 1.
- **Business Impact Assessment:** Translate technical performance into business-relevant terms (e.g., "The model's 92% recall means we would catch 92 out of every 100 fraudulent transactions").
- **Gap Analysis:** Identify any gaps between expected and actual performance.
- **Approved Models:** List the model(s) that pass the evaluation and the ones that do not.

**Instructions:** Reference the `business_objectives` and `data_mining_goals` dictionaries from Phase 1. Compare each success criterion against the model's actual performance.

In [9]:
metrics_path = Path("..") / "artifacts" / "model_metrics.json"
threshold_path = Path("..") / "artifacts" / "optimal_threshold.npy"

if metrics_path.exists():
    with open(metrics_path, "r", encoding="utf-8") as file:
        model_metrics = json.load(file)
    results_df = pd.DataFrame(model_metrics["results"])
    
    # Get the tuned model row (should be first after sorting)
    best_row = results_df.sort_values("f1_score", ascending=False).iloc[0].to_dict()
    
    # Load threshold if available
    optimal_threshold = None
    if threshold_path.exists():
        optimal_threshold = float(np.load(threshold_path))
else:
    model_metrics = None
    results_df = pd.DataFrame()
    best_row = {
        "model": "Pending execution",
        "accuracy": np.nan,
        "precision": np.nan,
        "recall": np.nan,
        "f1_score": np.nan,
    }
    optimal_threshold = None

evaluation = {
    "best_model": best_row["model"],
    "optimal_threshold": optimal_threshold,
    "business_criteria": [
        {
            "criterion": "Flag the majority of future attrition cases (target recall >= 0.60)",
            "observed_value": round(float(best_row["recall"]), 3) if pd.notna(best_row["recall"]) else "Run Phase 4",
            "status": "Met" if pd.notna(best_row["recall"]) and best_row["recall"] >= 0.60 else "Not yet met",
        },
        {
            "criterion": "Maintain usable precision for HR actioning (target precision >= 0.40)",
            "observed_value": round(float(best_row["precision"]), 3) if pd.notna(best_row["precision"]) else "Run Phase 4",
            "status": "Met" if pd.notna(best_row["precision"]) and best_row["precision"] >= 0.40 else "Not yet met",
        },
        {
            "criterion": "Provide a ranked risk model supported by interpretable drivers",
            "observed_value": "Supported by the selected feature set and baseline model comparison",
            "status": "Met",
        },
    ],
    "overall_assessment": (
        "Proceed toward a controlled pilot if the best model satisfies recall and precision targets; "
        "otherwise iterate on feature engineering and threshold tuning."
    ),
}

In [10]:
print("=" * 60)
print("EVALUATION AGAINST BUSINESS SUCCESS CRITERIA")
print("=" * 60)
print(f"Best model: {evaluation['best_model']}")
if evaluation['optimal_threshold']:
    print(f"Optimal threshold: {evaluation['optimal_threshold']:.3f}")

for item in evaluation["business_criteria"]:
    print(f"- {item['criterion']}")
    print(f"  Observed value: {item['observed_value']}")
    print(f"  Status: {item['status']}")

print()
print(evaluation["overall_assessment"])

EVALUATION AGAINST BUSINESS SUCCESS CRITERIA
Best model: Logistic Regression (SMOTE + Threshold Tuned)
Optimal threshold: 0.274
- Flag the majority of future attrition cases (target recall >= 0.60)
  Observed value: 0.617
  Status: Met
- Maintain usable precision for HR actioning (target precision >= 0.40)
  Observed value: 0.483
  Status: Met
- Provide a ranked risk model supported by interpretable drivers
  Observed value: Supported by the selected feature set and baseline model comparison
  Status: Met

Proceed toward a controlled pilot if the best model satisfies recall and precision targets; otherwise iterate on feature engineering and threshold tuning.


---
### Task 2: Review Process

Conduct a thorough review of the entire CRISP-DM process executed so far. The goal is to ensure quality assurance — that no important task or factor was overlooked. Consider:

- **Completeness Check:** Were all relevant data sources considered? Were the data preparation steps sufficient?
- **Methodology Review:** Were appropriate techniques used at each stage? Were there alternatives worth exploring?
- **Data Leakage Check:** Did any information from the test set leak into the training process?
- **Bias and Fairness:** Could the model introduce or amplify any biases? Is the model fair across different subgroups?
- **Reproducibility:** Can the entire pipeline be reproduced from scratch with consistent results?

**Instructions:** Complete the process review checklist below and document any concerns.

In [11]:
process_review = {
    "checklist": [
        {"item": "Business problem and stakeholders were defined", "status": "Yes", "notes": "Phase 1 documented objective, stakeholders, and success criteria."},
        {"item": "Relevant source data was profiled", "status": "Yes", "notes": "Phase 2 documented schema, class balance, and exploratory findings."},
        {"item": "Data quality issues were reviewed", "status": "Yes", "notes": "No material missing data; identifier and constant fields were removed."},
        {"item": "Feature engineering decisions were documented", "status": "Yes", "notes": "Derived tenure, promotion, commute, and travel features were added."},
        {"item": "Multiple candidate models were compared", "status": "Yes", "notes": "Phase 4 compares interpretable and ensemble models."},
        {"item": "Fairness and operational governance were fully assessed", "status": "Partial", "notes": "Further review is needed before production rollout."},
    ],
    "summary": "The CRISP-DM flow is substantially complete, with governance and pilot-design work remaining before production use.",
}


In [12]:
print("=== Process Review Checklist ===")
for item in process_review["checklist"]:
    print(f"[{item['status']:^7}] {item['item']}")
    print(f"Notes: {item['notes']}")

print()
print(process_review["summary"])


=== Process Review Checklist ===
[  Yes  ] Business problem and stakeholders were defined
Notes: Phase 1 documented objective, stakeholders, and success criteria.
[  Yes  ] Relevant source data was profiled
Notes: Phase 2 documented schema, class balance, and exploratory findings.
[  Yes  ] Data quality issues were reviewed
Notes: No material missing data; identifier and constant fields were removed.
[  Yes  ] Feature engineering decisions were documented
Notes: Derived tenure, promotion, commute, and travel features were added.
[  Yes  ] Multiple candidate models were compared
Notes: Phase 4 compares interpretable and ensemble models.
[Partial] Fairness and operational governance were fully assessed
Notes: Further review is needed before production rollout.

The CRISP-DM flow is substantially complete, with governance and pilot-design work remaining before production use.


---
### Task 3: Determine Next Steps

Based on the evaluation results and process review, decide on the next course of action. The three main options are:

1. **Proceed to Deployment:** The model meets all criteria and is ready for deployment.
2. **Iterate:** Go back to a previous phase (e.g., Data Preparation or Modelling) to improve the model before deployment.
3. **Terminate:** The project is not viable in its current form and should be re-scoped or abandoned.

**Instructions:** Document your decision and provide a clear rationale.

In [13]:
if pd.notna(best_row["recall"]) and pd.notna(best_row["precision"]) and best_row["recall"] >= 0.60 and best_row["precision"] >= 0.40:
    decision = "Proceed to Deployment"
    rationale = "The current best model meets the minimum screening thresholds for an HR pilot."
else:
    decision = "Iterate"
    rationale = "Model performance should be improved or threshold-tuned before operational use."

next_steps = {
    "decision": decision,
    "rationale": rationale,
    "if_iterating": {
        "return_to_phase": "Phase 3 or Phase 4",
        "actions": [
            "Experiment with threshold tuning to improve recall-precision balance.",
            "Add business-calibrated cost weighting for false negatives vs. false positives.",
            "Review feature engineering and fairness implications by employee subgroup.",
        ],
    },
    "if_deploying": {
        "deployment_mode": "Controlled HR analytics pilot",
        "actions": [
            "Share risk scores with HR partners only.",
            "Track intervention outcomes and feedback.",
            "Require human review before any employee action is taken.",
        ],
    },
}


In [14]:
print("=" * 60)
print("NEXT STEPS DECISION")
print("=" * 60)
print(f"Decision  : {next_steps['decision']}")
print(f"Rationale : {next_steps['rationale']}")

if next_steps["decision"] == "Iterate":
    print("\nRecommended iteration actions:")
    for action in next_steps["if_iterating"]["actions"]:
        print(f"- {action}")
else:
    print("\nRecommended deployment actions:")
    for action in next_steps["if_deploying"]["actions"]:
        print(f"- {action}")


NEXT STEPS DECISION
Decision  : Proceed to Deployment
Rationale : The current best model meets the minimum screening thresholds for an HR pilot.

Recommended deployment actions:
- Share risk scores with HR partners only.
- Track intervention outcomes and feedback.
- Require human review before any employee action is taken.
